# Analyze Author Name Mismatches

Quantifies the impact of improved name parsing in `CreateAuthorNames` on existing author-work assignments.

Uses `openalex.authors.author_names` with inline initial classification:
- Strip periods from parsed_name fields
- First name: length 1 after stripping → initial only
- Middle name: length <= 2 after stripping → initials only
- Derive first_initial as first letter of first name (same as MatchAuthors)

Split into two passes for efficiency:
- **Pass 1:** Block-key comparison (catches CJK reorders, ALL CAPS surname swaps)
- **Pass 2:** Full 8-pattern matching only on the subset where block keys match

**Warehouse:** Run on 4x-large — 790M work_author rows joined to 225M author_names.

---
## Pass 1: Block-Key Comparison

Compares the block_key derived from each work's `raw_author_name` against the block_key derived from the assigned author's `full_name`. A block_key mismatch means different last name or first initial — no matching pattern can succeed.

In [0]:
CREATE OR REPLACE TABLE openalex.authors.name_mismatch_block_key AS
WITH classified_names AS (
    SELECT
        raw_author_name,
        parsed_name,
        -- First name: strip periods, length 1 → initial only
        CASE
            WHEN parsed_name.first IS NULL OR TRIM(parsed_name.first) = '' THEN NULL
            WHEN LENGTH(REPLACE(REPLACE(parsed_name.first, '.', ''), ' ', '')) <= 1 THEN NULL
            ELSE REPLACE(parsed_name.first, '.', '')
        END AS normalized_first,
        CASE
            WHEN parsed_name.first IS NULL OR TRIM(parsed_name.first) = '' THEN NULL
            ELSE SUBSTRING(REPLACE(REPLACE(parsed_name.first, '.', ''), ' ', ''), 1, 1)
        END AS normalized_first_initial,
        -- Middle name: strip periods/spaces, length <= 2 → initials only
        CASE
            WHEN parsed_name.middle IS NULL OR TRIM(parsed_name.middle) = '' THEN NULL
            WHEN LENGTH(REPLACE(REPLACE(parsed_name.middle, '.', ''), ' ', '')) <= 2 THEN NULL
            ELSE REPLACE(parsed_name.middle, '.', '')
        END AS normalized_middle,
        CASE
            WHEN parsed_name.middle IS NULL OR TRIM(parsed_name.middle) = '' THEN NULL
            ELSE SUBSTRING(REPLACE(REPLACE(parsed_name.middle, '.', ''), ' ', ''), 1, 1)
        END AS normalized_middle_initials,
        -- Last name: just strip periods
        CASE
            WHEN parsed_name.last IS NULL OR TRIM(parsed_name.last) = '' THEN NULL
            ELSE REPLACE(parsed_name.last, '.', '')
        END AS normalized_last,
        -- Block key
        CASE
            WHEN parsed_name.last IS NULL OR TRIM(parsed_name.last) = '' THEN NULL
            WHEN parsed_name.first IS NULL OR TRIM(parsed_name.first) = '' THEN REPLACE(parsed_name.last, '.', '')
            ELSE CONCAT(
                SUBSTRING(REPLACE(REPLACE(parsed_name.first, '.', ''), ' ', ''), 1, 1),
                ' ',
                REPLACE(parsed_name.last, '.', '')
            )
        END AS block_key
    FROM openalex.authors.author_names
),
assigned_works AS (
    SELECT
        wa.work_id,
        wa.author_sequence,
        wa.author_id,
        wa.raw_author_name,
        a.full_name AS author_full_name
    FROM openalex.works.work_authors wa
    INNER JOIN openalex.authors.authors a ON wa.author_id = a.id
    WHERE wa.author_id IS NOT NULL
      AND wa.raw_author_name IS NOT NULL
      AND TRIM(wa.raw_author_name) != ''
)
SELECT
    aw.work_id,
    aw.author_sequence,
    aw.author_id,
    aw.raw_author_name,
    aw.author_full_name,
    wcn.block_key AS work_block_key,
    acn.block_key AS author_block_key,
    wcn.normalized_last AS w_last,
    wcn.normalized_first_initial AS w_first_initial,
    acn.normalized_last AS a_last,
    acn.normalized_first_initial AS a_first_initial
FROM assigned_works aw
LEFT JOIN classified_names wcn ON TRIM(aw.raw_author_name) = wcn.raw_author_name
LEFT JOIN classified_names acn ON TRIM(aw.author_full_name) = acn.raw_author_name

### Pass 1 Summary

In [0]:
SELECT
    COUNT(*) AS total_assignments,
    COUNT_IF(work_block_key IS NOT NULL AND author_block_key IS NOT NULL
             AND work_block_key = author_block_key) AS block_key_match,
    COUNT_IF(work_block_key IS NULL OR author_block_key IS NULL) AS unparseable,
    COUNT_IF(work_block_key IS NOT NULL AND author_block_key IS NOT NULL
             AND work_block_key != author_block_key) AS block_key_mismatch,
    ROUND(COUNT_IF(work_block_key IS NOT NULL AND author_block_key IS NOT NULL
                   AND work_block_key != author_block_key) * 100.0 / COUNT(*), 4) AS mismatch_pct
FROM openalex.authors.name_mismatch_block_key

### Pass 1: Mismatch type breakdown

In [0]:
SELECT
    CASE
        WHEN w_last IS NOT NULL AND a_last IS NOT NULL AND w_last != a_last
             AND w_first_initial IS NOT NULL AND a_first_initial IS NOT NULL AND w_first_initial != a_first_initial
            THEN 'both_swapped'
        WHEN w_last IS NOT NULL AND a_last IS NOT NULL AND w_last != a_last
            THEN 'last_name_changed'
        WHEN w_first_initial IS NOT NULL AND a_first_initial IS NOT NULL AND w_first_initial != a_first_initial
            THEN 'first_initial_changed'
        ELSE 'other'
    END AS mismatch_type,
    COUNT(*) AS count,
    COUNT(DISTINCT author_id) AS distinct_authors
FROM openalex.authors.name_mismatch_block_key
WHERE work_block_key IS NOT NULL AND author_block_key IS NOT NULL
  AND work_block_key != author_block_key
GROUP BY 1
ORDER BY count DESC

### Pass 1: Author-level impact

In [0]:
SELECT
    COUNT(DISTINCT author_id) AS affected_authors,
    SUM(mismatched_works) AS total_affected_works,
    PERCENTILE_APPROX(mismatched_works, ARRAY(0.25, 0.5, 0.75, 0.9, 0.99)) AS works_per_author_percentiles
FROM (
    SELECT author_id, COUNT(*) AS mismatched_works
    FROM openalex.authors.name_mismatch_block_key
    WHERE work_block_key IS NOT NULL AND author_block_key IS NOT NULL
      AND work_block_key != author_block_key
    GROUP BY author_id
)

### Pass 1: Author size distribution

In [0]:
SELECT
    CASE
        WHEN total_works <= 5 THEN '1-5'
        WHEN total_works <= 20 THEN '6-20'
        WHEN total_works <= 100 THEN '21-100'
        WHEN total_works <= 1000 THEN '101-1000'
        ELSE '1000+'
    END AS author_size_bucket,
    COUNT(*) AS author_count,
    SUM(mismatched_works) AS total_mismatched_works
FROM (
    SELECT
        nma.author_id,
        COALESCE(oa.works_count, 0) AS total_works,
        COUNT(*) AS mismatched_works
    FROM openalex.authors.name_mismatch_block_key nma
    LEFT JOIN openalex.authors.openalex_authors oa ON nma.author_id = oa.id
    WHERE nma.work_block_key IS NOT NULL AND nma.author_block_key IS NOT NULL
      AND nma.work_block_key != nma.author_block_key
    GROUP BY nma.author_id, oa.works_count
)
GROUP BY 1
ORDER BY 1

### Pass 1: Spot-check examples

In [0]:
SELECT
    raw_author_name, author_full_name,
    w_first_initial, w_last, work_block_key,
    a_first_initial, a_last, author_block_key,
    author_id
FROM openalex.authors.name_mismatch_block_key
WHERE work_block_key IS NOT NULL AND author_block_key IS NOT NULL
  AND work_block_key != author_block_key
LIMIT 50

---
## Pass 2: Full Pattern Matching (block-key-matching rows only)

For the subset where block keys match (same first initial + last name), run the full 8 name matching patterns to catch subtler mismatches. Uses inline initial classification from `author_names.parsed_name`.

In [0]:
CREATE OR REPLACE TABLE openalex.authors.name_mismatch_pattern AS
WITH classified_names AS (
    SELECT
        raw_author_name,
        parsed_name,
        -- First name: strip periods, length 1 → initial only
        CASE
            WHEN parsed_name.first IS NULL OR TRIM(parsed_name.first) = '' THEN NULL
            WHEN LENGTH(REPLACE(REPLACE(parsed_name.first, '.', ''), ' ', '')) <= 1 THEN NULL
            ELSE REPLACE(parsed_name.first, '.', '')
        END AS normalized_first,
        CASE
            WHEN parsed_name.first IS NULL OR TRIM(parsed_name.first) = '' THEN NULL
            ELSE SUBSTRING(REPLACE(REPLACE(parsed_name.first, '.', ''), ' ', ''), 1, 1)
        END AS normalized_first_initial,
        -- Middle name: strip periods/spaces, length <= 2 → initials only
        CASE
            WHEN parsed_name.middle IS NULL OR TRIM(parsed_name.middle) = '' THEN NULL
            WHEN LENGTH(REPLACE(REPLACE(parsed_name.middle, '.', ''), ' ', '')) <= 2 THEN NULL
            ELSE REPLACE(parsed_name.middle, '.', '')
        END AS normalized_middle,
        CASE
            WHEN parsed_name.middle IS NULL OR TRIM(parsed_name.middle) = '' THEN NULL
            ELSE SUBSTRING(REPLACE(REPLACE(parsed_name.middle, '.', ''), ' ', ''), 1, 1)
        END AS normalized_middle_initials,
        -- Last name: just strip periods
        CASE
            WHEN parsed_name.last IS NULL OR TRIM(parsed_name.last) = '' THEN NULL
            ELSE REPLACE(parsed_name.last, '.', '')
        END AS normalized_last,
        -- Block key
        CASE
            WHEN parsed_name.last IS NULL OR TRIM(parsed_name.last) = '' THEN NULL
            WHEN parsed_name.first IS NULL OR TRIM(parsed_name.first) = '' THEN REPLACE(parsed_name.last, '.', '')
            ELSE CONCAT(
                SUBSTRING(REPLACE(REPLACE(parsed_name.first, '.', ''), ' ', ''), 1, 1),
                ' ',
                REPLACE(parsed_name.last, '.', '')
            )
        END AS block_key
    FROM openalex.authors.author_names
),
block_key_matches AS (
    SELECT work_id, author_sequence, author_id, raw_author_name, author_full_name
    FROM openalex.authors.name_mismatch_block_key
    WHERE work_block_key IS NOT NULL AND author_block_key IS NOT NULL
      AND work_block_key = author_block_key
),
with_both_parses AS (
    SELECT
        bk.work_id,
        bk.author_sequence,
        bk.author_id,
        bk.raw_author_name,
        bk.author_full_name,
        wcn.normalized_first AS w_first,
        wcn.normalized_first_initial AS w_first_initial,
        wcn.normalized_middle AS w_middle,
        wcn.normalized_middle_initials AS w_middle_initials,
        wcn.normalized_last AS w_last,
        acn.normalized_first AS a_first,
        acn.normalized_first_initial AS a_first_initial,
        acn.normalized_middle AS a_middle,
        acn.normalized_middle_initials AS a_middle_initials,
        acn.normalized_last AS a_last
    FROM block_key_matches bk
    LEFT JOIN classified_names wcn ON TRIM(bk.raw_author_name) = wcn.raw_author_name
    LEFT JOIN classified_names acn ON TRIM(bk.author_full_name) = acn.raw_author_name
),
with_patterns AS (
    SELECT
        *,
        -- Pattern 1: Exact Full Name
        (w_first IS NOT NULL AND w_middle IS NOT NULL AND a_first IS NOT NULL AND a_middle IS NOT NULL
         AND w_first = a_first AND w_middle = a_middle AND w_last = a_last
        ) AS pattern_1,
        -- Pattern 2: Exact First, Middle Initial match
        (w_first IS NOT NULL AND w_middle IS NULL AND w_middle_initials IS NOT NULL
         AND a_first IS NOT NULL AND w_first = a_first AND w_last = a_last
         AND (a_middle_initials IS NULL OR SUBSTRING(w_middle_initials, 1, 1) = SUBSTRING(a_middle_initials, 1, 1))
        ) AS pattern_2,
        -- Pattern 3: Initials match to Full
        (w_first IS NULL AND w_first_initial IS NOT NULL AND w_middle_initials IS NOT NULL
         AND a_first IS NOT NULL AND a_middle_initials IS NOT NULL
         AND w_first_initial = a_first_initial
         AND SUBSTRING(w_middle_initials, 1, 1) = SUBSTRING(a_middle_initials, 1, 1)
         AND w_last = a_last
        ) AS pattern_3,
        -- Pattern 4: Both have only initials
        (w_first IS NULL AND w_first_initial IS NOT NULL AND a_first IS NULL AND a_first_initial IS NOT NULL
         AND w_middle IS NULL AND w_middle_initials IS NOT NULL AND a_middle IS NULL AND a_middle_initials IS NOT NULL
         AND w_first_initial = a_first_initial
         AND SUBSTRING(w_middle_initials, 1, 1) = SUBSTRING(a_middle_initials, 1, 1)
         AND w_last = a_last
        ) AS pattern_4,
        -- Pattern 5: Exact First + Last, no middle
        (w_first IS NOT NULL AND a_first IS NOT NULL
         AND w_first = a_first AND w_last = a_last
         AND w_middle_initials IS NULL
        ) AS pattern_5,
        -- Pattern 6: First Initial to Full
        (w_first IS NULL AND w_first_initial IS NOT NULL AND w_middle_initials IS NULL
         AND a_first IS NOT NULL
         AND w_first_initial = a_first_initial AND w_last = a_last
        ) AS pattern_6,
        -- Pattern 7: Both have only first initial, no middle
        (w_first IS NULL AND w_first_initial IS NOT NULL AND a_first IS NULL AND a_first_initial IS NOT NULL
         AND w_middle_initials IS NULL AND a_middle_initials IS NULL
         AND w_first_initial = a_first_initial AND w_last = a_last
        ) AS pattern_7,
        -- Pattern 8: Full Name to Initial
        (w_first IS NOT NULL AND a_first IS NULL AND a_first_initial IS NOT NULL
         AND w_first_initial = a_first_initial AND w_last = a_last
        ) AS pattern_8
    FROM with_both_parses
)
SELECT
    work_id, author_sequence, author_id, raw_author_name, author_full_name,
    w_first, w_first_initial, w_middle, w_middle_initials, w_last,
    a_first, a_first_initial, a_middle, a_middle_initials, a_last,
    (pattern_1 OR pattern_2 OR pattern_3 OR pattern_4 OR pattern_5
     OR pattern_6 OR pattern_7 OR pattern_8) AS any_pattern_matches,
    pattern_1, pattern_2, pattern_3, pattern_4, pattern_5, pattern_6, pattern_7, pattern_8
FROM with_patterns
WHERE NOT (pattern_1 OR pattern_2 OR pattern_3 OR pattern_4 OR pattern_5
           OR pattern_6 OR pattern_7 OR pattern_8)

### Pass 2 Summary

In [0]:
SELECT
    COUNT(*) AS pattern_level_mismatches,
    COUNT(DISTINCT author_id) AS affected_authors
FROM openalex.authors.name_mismatch_pattern

### Pass 2: Spot-check examples

In [0]:
SELECT
    raw_author_name, author_full_name,
    w_first, w_middle, w_middle_initials,
    a_first, a_middle, a_middle_initials,
    w_last, a_last,
    author_id
FROM openalex.authors.name_mismatch_pattern
LIMIT 50

---
## Combined Summary

Total mismatches across both passes.

In [0]:
SELECT 'block_key_mismatch' AS source, COUNT(*) AS works, COUNT(DISTINCT author_id) AS authors
FROM openalex.authors.name_mismatch_block_key
WHERE work_block_key IS NOT NULL AND author_block_key IS NOT NULL
  AND work_block_key != author_block_key
UNION ALL
SELECT 'pattern_level_mismatch', COUNT(*), COUNT(DISTINCT author_id)
FROM openalex.authors.name_mismatch_pattern
UNION ALL
SELECT 'total',
    (SELECT COUNT(*) FROM openalex.authors.name_mismatch_block_key
     WHERE work_block_key IS NOT NULL AND author_block_key IS NOT NULL AND work_block_key != author_block_key)
    + (SELECT COUNT(*) FROM openalex.authors.name_mismatch_pattern),
    NULL